# Codex vs Kimi — Comparative Experiment Analysis

Head-to-head comparison of Codex (gpt-5.4) and Kimi Code (kimi-for-coding) as autonomous researchers.
Both agents run the same pipeline on the same seeds with the same reviewers.

## 1. Data Collection

In [ ]:
import jsonimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport matplotlibfrom collections import defaultdict, Counterfrom pathlib import Pathmatplotlib.rcParams['figure.figsize'] = (14, 6)matplotlib.rcParams['font.size'] = 11BASE = Path('..') if Path('../results').exists() else Path('.')def normalize_seed_name(raw):    s = raw    parts = s.rsplit('_', 2)    if len(parts) >= 3 and parts[-2].isdigit() and parts[-1].isdigit():        s = '_'.join(parts[:-2])    return (s.replace('_gpu', '').replace('_cpu', '')             .replace('_', ' ').title()             .replace('Ai For', 'AI for')             .replace('Of Learned Representations', 'of Learned Repr.')             .replace('And Benchmarks', '& Benchmarks')             .replace('And Cleaning', '& Cleaning')             .replace('In Machine Learning', 'in ML')             .replace('Representation Learning', 'Repr. Learning'))def get_trial_key(path_str, agent):    after = path_str.split(f'/{agent}/')[1].split('/')    if '/results/' in path_str:        return (normalize_seed_name(after[0]), after[1])    elif after[0].startswith('run_'):        return (normalize_seed_name(after[1]), after[0].replace('run_', 'trial'))    else:        return (normalize_seed_name(after[0]), 'trial1')rows = []for agent in ['codex', 'kimi']:    seen = set()        # results/ directory    agent_dir = BASE / 'results' / agent    if agent_dir.exists():        for seed_dir in sorted(agent_dir.iterdir()):            if not seed_dir.is_dir(): continue            platform = 'gpu' if '_gpu' in seed_dir.name else 'cpu'            seed = normalize_seed_name(seed_dir.name)            for trial_dir in sorted(seed_dir.iterdir()):                if not trial_dir.is_dir() or not trial_dir.name.startswith('trial'): continue                key = (seed, trial_dir.name)                if key in seen: continue                seen.add(key)                reviews = sorted(trial_dir.glob('code/idea_*/reviews.json'))                if not reviews: continue                try:                    r = json.loads(reviews[-1].read_text())                except: continue                row = {'agent': agent, 'seed': seed, 'platform': platform,                       'trial': int(trial_dir.name.replace('trial','')),                       'avg_score': r.get('avg_score', 0)}                for rev in r.get('reviews', []):                    name = rev.get('source', '?').replace('agent:', '')                    row[f'reviewer_{name}'] = rev.get('overall_score')                    for dim, val in rev.get('scores', {}).items():                        if isinstance(val, (int, float)):                            row[f'dim_{dim}'] = row.get(f'dim_{dim}', 0) + val                                # Tracker                tracker_file = trial_dir / 'tracker' / 'tracker.json'                if tracker_file.exists():                    try:                        tracker = json.loads(tracker_file.read_text())                        for a in tracker.get('actions', []):                            stage = a.get('stage', '')                            if 'self_review' in stage:                                k = f'sr_{stage}_count'                                row[k] = row.get(k, 0) + 1                            elapsed = a.get('elapsed_seconds', 0)                            if elapsed > 0:                                row[f'time_{stage}'] = row.get(f'time_{stage}', 0) + elapsed                    except: pass                summary_file = trial_dir / 'tracker' / 'summary.json'                if summary_file.exists():                    try:                        s = json.loads(summary_file.read_text())                        row['wall_time_h'] = s.get('wall_time_seconds', 0) / 3600                        row['ideas_tried'] = s.get('ideas_tried', 0)                    except: pass                rows.append(row)        # outputs/run_N/    for run_dir in sorted((BASE / 'outputs' / agent).glob('run_*')):        run_num = int(run_dir.name.split('_')[1])        for seed_dir in sorted(run_dir.iterdir()):            if not seed_dir.is_dir(): continue            seed = normalize_seed_name(seed_dir.name)            key = (seed, f'trial{run_num}')            if key in seen: continue            seen.add(key)            reviews = sorted(seed_dir.glob('idea_*/reviews.json'))            if not reviews: continue            try:                r = json.loads(reviews[-1].read_text())            except: continue            row = {'agent': agent, 'seed': seed, 'platform': 'gpu',                   'trial': run_num, 'avg_score': r.get('avg_score', 0)}            for rev in r.get('reviews', []):                name = rev.get('source', '?').replace('agent:', '')                row[f'reviewer_{name}'] = rev.get('overall_score')                for dim, val in rev.get('scores', {}).items():                    if isinstance(val, (int, float)):                        row[f'dim_{dim}'] = row.get(f'dim_{dim}', 0) + val            tracker_file = seed_dir / 'tracker.json'            if tracker_file.exists():                try:                    tracker = json.loads(tracker_file.read_text())                    for a in tracker.get('actions', []):                        stage = a.get('stage', '')                        if 'self_review' in stage:                            k = f'sr_{stage}_count'                            row[k] = row.get(k, 0) + 1                        elapsed = a.get('elapsed_seconds', 0)                        if elapsed > 0:                            row[f'time_{stage}'] = row.get(f'time_{stage}', 0) + elapsed                except: pass            summary_file = seed_dir / 'summary.json'            if summary_file.exists():                try:                    s = json.loads(summary_file.read_text())                    row['wall_time_h'] = s.get('wall_time_seconds', 0) / 3600                    row['ideas_tried'] = s.get('ideas_tried', 0)                except: pass            rows.append(row)    # outputs/timestamp    for ws in sorted((BASE / 'outputs' / agent).iterdir()):        if not ws.is_dir() or ws.name.startswith('run_') or ws.name == 'logs': continue        parts = ws.name.rsplit('_', 2)        if len(parts) < 3 or not parts[-2].isdigit(): continue        seed = normalize_seed_name(ws.name)        key = (seed, 'trial1')        if key in seen: continue        seen.add(key)        reviews = sorted(ws.glob('idea_*/reviews.json'))        if not reviews: continue        try:            r = json.loads(reviews[-1].read_text())        except: continue        row = {'agent': agent, 'seed': seed, 'platform': 'gpu',               'trial': 1, 'avg_score': r.get('avg_score', 0)}        for rev in r.get('reviews', []):            name = rev.get('source', '?').replace('agent:', '')            row[f'reviewer_{name}'] = rev.get('overall_score')        rows.append(row)df = pd.DataFrame(rows)print(f'Total trials: {len(df)}')for agent in ['codex', 'kimi']:    ad = df[df.agent == agent]    print(f'  {agent}: {len(ad)} trials ({len(ad[ad.platform=="cpu"])} CPU, {len(ad[ad.platform=="gpu"])} GPU)')

## 2. Overall Score Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))# Overallax = axes[0]for agent, color in [('codex', '#4CAF50'), ('kimi', '#FF9800')]:    scores = df[df.agent == agent]['avg_score']    ax.hist(scores, bins=np.arange(0, 10.5, 0.5), alpha=0.5, color=color,            edgecolor='white', label=f'{agent.title()} (μ={scores.mean():.2f}±{scores.sem():.2f})')ax.set_xlabel('Peer Review Score')ax.set_ylabel('Count')ax.set_title('All Trials')ax.legend()# CPU onlyax = axes[1]for agent, color in [('codex', '#4CAF50'), ('kimi', '#FF9800')]:    scores = df[(df.agent == agent) & (df.platform == 'cpu')]['avg_score']    if len(scores) > 0:        ax.hist(scores, bins=np.arange(0, 10.5, 0.5), alpha=0.5, color=color,                edgecolor='white', label=f'{agent.title()} (μ={scores.mean():.2f})')ax.set_xlabel('Peer Review Score')ax.set_title('CPU Seeds Only')ax.legend()# GPU onlyax = axes[2]for agent, color in [('codex', '#4CAF50'), ('kimi', '#FF9800')]:    scores = df[(df.agent == agent) & (df.platform == 'gpu')]['avg_score']    if len(scores) > 0:        ax.hist(scores, bins=np.arange(0, 10.5, 0.5), alpha=0.5, color=color,                edgecolor='white', label=f'{agent.title()} (μ={scores.mean():.2f})')ax.set_xlabel('Peer Review Score')ax.set_title('GPU Seeds Only')ax.legend()plt.suptitle('Codex vs Kimi — Score Distributions', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('comparison_score_distributions.png', dpi=150, bbox_inches='tight')plt.show()# Summary tablesummary = df.groupby(['agent', 'platform'])['avg_score'].agg(['mean', 'std', 'count']).round(2)summary['se'] = (summary['std'] / np.sqrt(summary['count'])).round(2)print(summary)

## 3. Head-to-Head by Seed

In [ ]:
seed_means = df.groupby(['agent', 'platform', 'seed'])['avg_score'].agg(['mean', 'sem']).reset_index()for platform in ['cpu', 'gpu']:    pdata = seed_means[seed_means.platform == platform]    seeds = sorted(pdata.seed.unique())        fig, ax = plt.subplots(figsize=(14, max(6, len(seeds) * 0.6)))    y = np.arange(len(seeds))    w = 0.35        for i, (agent, color) in enumerate([('codex', '#4CAF50'), ('kimi', '#FF9800')]):        agent_data = pdata[pdata.agent == agent]        means = [agent_data[agent_data.seed == s]['mean'].values[0] if s in agent_data.seed.values else 0 for s in seeds]        sems = [agent_data[agent_data.seed == s]['sem'].values[0] if s in agent_data.seed.values else 0 for s in seeds]        ax.barh(y + i * w, means, w, xerr=sems, color=color, alpha=0.8, capsize=3,                label=f'{agent.title()}')        ax.set_yticks(y + w/2)    ax.set_yticklabels(seeds, fontsize=9)    ax.set_xlabel('Peer Review Score (mean ± SE)')    ax.set_title(f'Codex vs Kimi — {platform.upper()} Seeds')    ax.legend()    ax.axvline(x=8, color='green', linestyle='--', alpha=0.3, label='Accept')    ax.set_xlim(0, 9)        plt.tight_layout()    plt.savefig(f'comparison_seeds_{platform}.png', dpi=150, bbox_inches='tight')    plt.show()

## 4. Reviewer Scoring by Agent

In [ ]:
reviewers = ['Claude Code', 'Codex', 'Kimi Code']fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, reviewer in zip(axes, reviewers):    col = f'reviewer_{reviewer}'    if col not in df.columns: continue    for agent, color in [('codex', '#4CAF50'), ('kimi', '#FF9800')]:        scores = df[df.agent == agent][col].dropna()        ax.hist(scores, bins=np.arange(-0.5, 11.5, 1), alpha=0.5, color=color,                edgecolor='white', label=f'{agent.title()} (μ={scores.mean():.1f})')    ax.set_xlabel('Score')    ax.set_title(f'{reviewer} Reviews')    ax.legend()plt.suptitle('How Each Reviewer Scores Codex vs Kimi Papers', fontsize=13)plt.tight_layout()plt.savefig('comparison_reviewer_scores.png', dpi=150, bbox_inches='tight')plt.show()

## 5. Per-Dimension Comparison

In [ ]:
dimensions = ['novelty', 'soundness', 'significance', 'clarity', 'reproducibility',              'experimental_rigor', 'references', 'reference_integrity', 'results_integrity']agent_dims = {}for agent in ['codex', 'kimi']:    means = []    for dim in dimensions:        col = f'dim_{dim}'        if col in df.columns:            vals = df[df.agent == agent][col].dropna()            # Divide by number of reviewers (3) since we summed them            means.append(vals.mean() / 3 if len(vals) > 0 else 0)        else:            means.append(0)    agent_dims[agent] = meansfig, ax = plt.subplots(figsize=(14, 6))x = np.arange(len(dimensions))w = 0.35for i, (agent, color) in enumerate([('codex', '#4CAF50'), ('kimi', '#FF9800')]):    ax.bar(x + i * w, agent_dims[agent], w, color=color, alpha=0.8, label=agent.title())ax.set_xticks(x + w/2)ax.set_xticklabels([d.replace('_', '\n') for d in dimensions], fontsize=9)ax.set_ylabel('Mean Score (1-10)')ax.set_title('Codex vs Kimi — Per-Dimension Scores')ax.legend()ax.axhline(y=6, color='gray', linestyle='--', alpha=0.3)plt.tight_layout()plt.savefig('comparison_dimensions.png', dpi=150, bbox_inches='tight')plt.show()

## 6. Score Consistency (Variance Across Trials)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))seed_std = df.groupby(['agent', 'seed'])['avg_score'].std().reset_index()seed_std.columns = ['agent', 'seed', 'std']for agent, color in [('codex', '#4CAF50'), ('kimi', '#FF9800')]:    ad = seed_std[seed_std.agent == agent].sort_values('std')    ax.scatter(ad['seed'], ad['std'], c=color, s=80, alpha=0.7, label=f'{agent.title()} (mean std={ad["std"].mean():.2f})')ax.set_ylabel('Std Dev Across Trials')ax.set_title('Score Consistency — Lower is More Consistent')ax.legend()plt.xticks(rotation=45, ha='right', fontsize=8)plt.tight_layout()plt.savefig('comparison_consistency.png', dpi=150, bbox_inches='tight')plt.show()print(f"Codex mean std: {seed_std[seed_std.agent=='codex']['std'].mean():.3f}")print(f"Kimi mean std:  {seed_std[seed_std.agent=='kimi']['std'].mean():.3f}")

## 7. CPU vs GPU Performance Gap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))data = []for agent in ['codex', 'kimi']:    for platform in ['cpu', 'gpu']:        scores = df[(df.agent == agent) & (df.platform == platform)]['avg_score']        if len(scores) > 0:            data.append({'agent': agent.title(), 'platform': platform.upper(),                        'mean': scores.mean(), 'se': scores.sem()})plot_df = pd.DataFrame(data)x = np.arange(2)w = 0.35for i, platform in enumerate(['CPU', 'GPU']):    pdata = plot_df[plot_df.platform == platform]    ax.bar(x + i * w, pdata['mean'], w, yerr=pdata['se'],           color=['#2196F3', '#FF9800'][i], alpha=0.7, capsize=5,           label=platform)ax.set_xticks(x + w/2)ax.set_xticklabels(plot_df.agent.unique())ax.set_ylabel('Mean Peer Review Score')ax.set_title('CPU vs GPU Performance by Agent')ax.legend()ax.axhline(y=8, color='green', linestyle='--', alpha=0.3)plt.tight_layout()plt.savefig('comparison_cpu_gpu_gap.png', dpi=150, bbox_inches='tight')plt.show()for agent in ['codex', 'kimi']:    cpu = df[(df.agent == agent) & (df.platform == 'cpu')]['avg_score'].mean()    gpu = df[(df.agent == agent) & (df.platform == 'gpu')]['avg_score'].mean()    print(f"{agent.title()}: CPU={cpu:.2f}, GPU={gpu:.2f}, gap={cpu-gpu:+.2f}")

## 8. Wall Time Comparison

In [ ]:
wt = df[df['wall_time_h'] > 0]fig, ax = plt.subplots(figsize=(10, 5))for agent, color in [('codex', '#4CAF50'), ('kimi', '#FF9800')]:    times = wt[wt.agent == agent]['wall_time_h']    if len(times) > 0:        ax.hist(times, bins=15, alpha=0.5, color=color,                edgecolor='white', label=f'{agent.title()} (μ={times.mean():.1f}h)')ax.set_xlabel('Wall Time (hours)')ax.set_ylabel('Count')ax.set_title('Wall Time Distribution Per Run')ax.legend()plt.tight_layout()plt.savefig('comparison_wall_time.png', dpi=150, bbox_inches='tight')plt.show()

## 8b. Time Per Stage Comparison

How much time does each agent spend on each pipeline stage?

In [ ]:
# Collect time per stage from tracker datastage_times = defaultdict(lambda: defaultdict(list))for agent in ['codex', 'kimi']:    for tracker_file in list((BASE / 'results' / agent).glob('*/trial*/tracker/tracker.json')) + \                         list((BASE / 'outputs' / agent).glob('run_*/*/tracker.json')):        try:            tracker = json.loads(tracker_file.read_text())            for a in tracker.get('actions', []):                stage = a.get('stage', '')                elapsed = a.get('elapsed_seconds', 0)                if elapsed > 0 and stage:                    stage_times[stage][agent].append(elapsed / 60)  # minutes        except: pass# Filter to main stagesmain_stages = ['ideation', 'self_review_idea', 'experiments', 'self_review_experiment',               'paper', 'self_review_paper', 'review']stage_labels = ['Ideation', 'Idea\nReview', 'Experiments', 'Exp\nReview',                'Paper', 'Paper\nReview', 'Peer\nReview']fig, axes = plt.subplots(1, 2, figsize=(18, 6))# Bar chart: mean time per stageax = axes[0]x = np.arange(len(main_stages))w = 0.35for i, (agent, color) in enumerate([('codex', '#4CAF50'), ('kimi', '#FF9800')]):    means = [np.mean(stage_times[s][agent]) if stage_times[s][agent] else 0 for s in main_stages]    sems = [np.std(stage_times[s][agent]) / np.sqrt(len(stage_times[s][agent]))            if len(stage_times[s][agent]) > 1 else 0 for s in main_stages]    bars = ax.bar(x + i * w, means, w, yerr=sems, color=color, alpha=0.8, capsize=3,                  label=agent.title())    for bar, m in zip(bars, means):        if m > 0:            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,                    f'{m:.0f}', ha='center', fontsize=7)ax.set_xticks(x + w/2)ax.set_xticklabels(stage_labels, fontsize=9)ax.set_ylabel('Mean Time (minutes)')ax.set_title('Mean Time Per Stage')ax.legend()# Total time breakdown (stacked bar)ax = axes[1]bottom_codex = np.zeros(1)bottom_kimi = np.zeros(1)colors = plt.cm.tab10(np.linspace(0, 1, len(main_stages)))for j, (stage, label) in enumerate(zip(main_stages, stage_labels)):    for i, agent in enumerate(['codex', 'kimi']):        total = sum(stage_times[stage][agent])        n = max(1, len(set(  # approximate unique trials            str(f) for f in (BASE / 'results' / agent).glob('*/trial*/tracker/tracker.json')        )))        avg_total = total / max(1, n) if n > 0 else 0        bottom = bottom_codex if agent == 'codex' else bottom_kimi        ax.barh(i, avg_total, left=bottom[0], color=colors[j], alpha=0.8,                label=label.replace('\n', ' ') if i == 0 else '')        if avg_total > 5:            ax.text(bottom[0] + avg_total/2, i, f'{avg_total:.0f}m', ha='center',                    va='center', fontsize=7)        bottom[0] += avg_totalax.set_yticks([0, 1])ax.set_yticklabels(['Codex', 'Kimi'])ax.set_xlabel('Total Time Per Run (minutes)')ax.set_title('Time Breakdown Per Average Run')ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)plt.suptitle('Codex vs Kimi — Time Per Stage', fontsize=13, fontweight='bold')plt.tight_layout()plt.savefig('comparison_time_per_stage.png', dpi=150, bbox_inches='tight')plt.show()# Print tableprint(f'{"Stage":<25} | {"Codex (min)":<20} | {"Kimi (min)":<20} | {"Ratio"}')print('-'*25 + '-|' + '-'*22 + '|' + '-'*22 + '|' + '-'*10)for stage, label in zip(main_stages, [l.replace('\n', ' ') for l in stage_labels]):    c = stage_times[stage]['codex']    k = stage_times[stage]['kimi']    cm = np.mean(c) if c else 0    km = np.mean(k) if k else 0    ratio = f'{cm/km:.1f}x' if km > 0 else 'n/a'    print(f'{label:<25} | {cm:>6.1f} (n={len(c):<4}) | {km:>6.1f} (n={len(k):<4}) | {ratio}')

## 9. Score Heatmap (Agent × Seed)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))for ax, agent in zip(axes, ['codex', 'kimi']):    ad = df[df.agent == agent]    pivot = ad.pivot_table(values='avg_score', index='seed', columns='trial', aggfunc='first')    pivot = pivot.sort_index()        im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=8)    ax.set_xticks(range(len(pivot.columns)))    ax.set_xticklabels([f'Trial {c}' for c in pivot.columns])    ax.set_yticks(range(len(pivot.index)))    ax.set_yticklabels(pivot.index, fontsize=8)    ax.set_title(f'{agent.title()}')        for i in range(len(pivot.index)):        for j in range(len(pivot.columns)):            val = pivot.iloc[i, j]            if not np.isnan(val):                color = 'white' if val < 3 else 'black'                ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9, color=color)for ax in axes:    fig.colorbar(ax.images[0], ax=ax, label='Score', shrink=0.8)fig.suptitle('Codex vs Kimi — Score Heatmap (Seed × Trial)', fontsize=13)plt.tight_layout()plt.savefig('comparison_heatmap.png', dpi=150, bbox_inches='tight')plt.show()

## 10. Summary Statistics

In [ ]:
print('=== CODEX vs KIMI SUMMARY ===')for agent in ['codex', 'kimi']:    ad = df[df.agent == agent]    print(f'\n{agent.title()}:')    print(f'  Total trials: {len(ad)} ({len(ad[ad.platform=="cpu"])} CPU, {len(ad[ad.platform=="gpu"])} GPU)')    print(f'  Overall: {ad.avg_score.mean():.2f} ± {ad.avg_score.sem():.2f}')    print(f'  CPU:     {ad[ad.platform=="cpu"].avg_score.mean():.2f} ± {ad[ad.platform=="cpu"].avg_score.sem():.2f}')    print(f'  GPU:     {ad[ad.platform=="gpu"].avg_score.mean():.2f} ± {ad[ad.platform=="gpu"].avg_score.sem():.2f}')    print(f'  Best:    {ad.avg_score.max():.1f}')    print(f'  >= 5.0:  {len(ad[ad.avg_score >= 5])} / {len(ad)} ({100*len(ad[ad.avg_score >= 5])/len(ad):.0f}%)')print(f'\nCodex - Kimi gap: {df[df.agent=="codex"].avg_score.mean() - df[df.agent=="kimi"].avg_score.mean():+.2f}')